## RAG pipeline - Data ingestion to vector DB pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [6]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf")) ## why **, so its recursive and we get subfolderstoo
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try: 
            loader = PyPDFLoader(str(pdf_file)) #why try and except for each file, so we get correupted file only
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: embeddings.pdf
  ✓ Loaded 12 pages

Processing: attention.pdf
  ✓ Loaded 15 pages

Processing: yolo.pdf
  ✓ Loaded 10 pages

Total documents loaded: 37


### Chunking

In [7]:

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 37 documents into 160 chunks

Example chunk:
Content: Efﬁcient Estimation of Word Representations in
Vector Space
Tomas Mikolov
Google Inc., Mountain View, CA
tmikolov@google.com
Kai Chen
Google Inc., Mountain View, CA
kaichen@google.com
Greg Corrado
Goo...
Metadata: {'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2013-09-10T00:03:46+00:00', 'author': '', 'keywords': '', 'moddate': '2013-09-10T00:03:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.12', 'creator': 'LaTeX with hyperref package', 'creationdate': '2013-09-10T00:03:46+00:00', 'author': '', 'keywords': '', 'moddate': '2013-09-10T00:03:46+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.1415926-2.3-1.40.12 (TeX Live 2011) kpathsea version 6.0.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}, page_content='Efﬁcient Estimation of Word Representations in\nVector Space\nTomas Mikolov\nGoogle Inc., Mountain View, CA\ntmikolov@google.com\nKai Chen\nGoogle Inc., Mountain View, CA\nkaichen@google.com\nGreg Corrado\nGoogle Inc., Mountain View, CA\ngcorrado@google.com\nJeffrey Dean\nGoogle Inc., Mountain View, CA\njeff@google.com\nAbstract\nWe propose two novel model architectures for computing continuous vector repre-\nsentations of words from very large data sets. The quality of these

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/yashsinghal/Desktop/Agentic Ai/Traditional Rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:

from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

        print(
            f"Loaded {model_name}. "
            f"Embedding dimension: {self.model.get_embedding_dimension()}"
        )

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        embeddings = self.model.encode(texts, show_progress_bar=True)

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6703.60it/s]


Loaded all-MiniLM-L6-v2. Embedding dimension: 384


/var/folders/q7/qxk8vry10nq7nhvkz9c8t2680000gn/T/ipykernel_15971/2003137909.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


### VectorStore


In [16]:

import os
import uuid
from typing import List, Any

import chromadb
import numpy as np


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        os.makedirs(persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
        )

        print(f"Vector store initialized: {collection_name}")
        print(f"Existing documents: {self.collection.count()}")

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add documents and their embeddings to ChromaDB."""

        if len(documents) != len(embeddings): #VVV important
            raise ValueError("Number of documents and embeddings must match")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4().hex[:8]}_{i}")

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
        )

        print(f"Added {len(documents)} documents")
        print(f"Total documents: {self.collection.count()}")


vectorstore = VectorStore()



Vector store initialized: pdf_documents
Existing documents: 0


- Why PersistentClient and not the default in-memory client — persistence means your embeddings survive a restart; without it, you'd re-embed everything every time you run the app, which is slow and wasteful

- Why .tolist() on the embeddings — ChromaDB's API expects JSON-serializable lists, not numpy arrays; numpy arrays aren't directly serializable, so this conversion is mandatory, not stylistic

- Why the len(documents) != len(embeddings) check exists — a silent length mismatch would corrupt your index (wrong embedding paired with wrong text) with no error until retrieval quietly returns garbage later — this is a real bug class in RAG systems, catching it early is the right instinct

- Why metadata gets doc_index and content_length added — these become filterable/debuggable fields later (e.g., "show me all chunks under 50 characters" to catch bad splitting), not just decoration

In [17]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 5/5 [00:02<00:00,  2.12it/s]


Generated embeddings with shape: (160, 384)
Added 160 documents
Total documents: 160


## Retriever Pipeline From VectorStore

In [18]:

from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        # Convert the user's query into an embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search ChromaDB for similar embeddings
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:

            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]

            for i, (doc_id, document, metadata, distance) in enumerate(
                zip(ids, documents, metadatas, distances)
            ):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": i + 1
                    })

        return retrieved_docs


rag_retriever = RAGRetriever(vectorstore, embedding_manager)



In [19]:
rag_retriever

In [20]:
rag_retriever.retrieve("What is attention is all you need")

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Generated embeddings with shape: (1, 384)


[{'id': 'doc_4c4e3e88_65',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'source': '../data/pdf/attention.pdf',
   'trapped': '/False',
   'keywords': '',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'page_label': '3',
   'doc_index': 65,
   'title': '',
   'producer': 'pdfTeX-1.40.25',
   'moddate': '2024-04-10T21:11:43+00:00',
   'page': 2,
   'content_length': 216,
   'total_pages': 15,
   'file_type': 'pdf',
   'creator': 'LaTeX with hyperref',
   'subject': '',
   'author': '',
   'source_file': 'attention.pdf',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'},
  'similarity_score': 0.1399548053741455,
  'distance': 0.8600451946258545,
  'rank': 1}]

In [22]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.97it/s]

Generated embeddings with shape: (1, 384)


[]